# TP 2 - Séance 2
## Apprentissage profond appliqué aux images médicales

Équipe #0
- Nom, Prénom (Matricule)
- Nom, Prénom (Matricule)

In [1]:
import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.manifold import TSNE
from tqdm import tqdm


In [2]:
from utils import get_dataloaders, get_embeddings, get_model_and_optimizer, train_epoch, validate


## Exercice 2.1: Exploration des données

Le jeu de données `dataset` est un objet de type [`DatasetDict`](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.DatasetDict) organisé comme un `dict` standard en Python, avec des entrées nommées `"train"` et `"test"` et qui contiennent des [`Dataset`](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset) - qui eux contiennent les images et leur label associé.

Vous pouvez interagir avec un `DatasetDict` essentiellement de la même manière qu'avec un `dict` (e.g. les méthodes `keys()`/`values()`/`items()`, aller chercher une entrée avec la notation par crochets `dataset["train"]`, etc.).

Pour interagir avec un `Dataset`, vous pouvez soit aller chercher un exemple donné (e.g. `dataset["train"][0]` qui est un `dict` qui contient une image et son label), soit aller chercher **tous** les labels/images du dataset (e.g. `list(dataset["train"]["label"])`).

<span style="color:red">**Attention!** Si vous essayez de charger toutes les images d'un coup, ça ne machera probablement pas car il y en a beaucoup trop pour la quantité de RAM qui est vraisemblablement disponible sur votre machine.
</span>

In [ ]:
dataset = datasets.load_dataset("zacharielegault/Kermany2017-OCT")

# Certaines images sont encodées par erreur comme RGB. On les convertit en grayscale.
dataset["test"] = dataset["test"].map(lambda d: {"image": d["image"].convert("L")})

# Les images sont par défaut encodées en images PIL. On préfère des arrays Numpy qu'on connait bien.
dataset["train"] = dataset["train"].with_format("numpy")
dataset["test"] = dataset["test"].with_format("numpy")

**Q1.** Combien d'images y a-t-il dans chaque sous-ensemble du dataset?

**Q2.** Quelles sont les classes représentées dans le dataset? Comment sont-elles représentées?

NB: Vous pouvez utiliser les méthodes `dataset["train"].features["label"].int2str(...)` et `dataset["train"].features["label"].str2int(...)` pour identifier les correspondances.

**Q3.** Quel nombre et proportion des images de chaque sous-ensemble appartient aux différentes classes? Affichez un histogramme de la distribution des classes pour chaque sous-ensemble.

**Q4.** Affichez 2 échantillons de chaque classe. Pour chaque image, affichez l'indice et la classe dans le titre. Décrivez les différences que vous observez entre les classes.

**Q5.** Vous remarquerez que les images ne sont pas toutes de la même taille. Identifiez les tailles d'images et leur fréquence dans chaque sous-ensemble. Que pouvez-vous observer sur leur distribution? En quoi est-ce que des images de tailles différentes peut être un problème?

## Exercice 2.2: Entraînement d'un modèle de classification

**Q1**. Créez un modèle qui n'a pas été pré-entraîné (utilisez la valeur par défaut de `pretrained=False`). Calculez la _loss_ initiale du modèle sur 10% des données d'entraînement. Est-ce que la _loss_ initiale est bien ce à quoi vous vous attendiez?

In [9]:
train_loader, val_loader, test_loader = get_dataloaders(dataset, batch_size=64, train_subset=0.1)

**Q2**. Entraînez votre modèle sur 10% des données d'entraînement (20 époques devraient suffire). Affichez la courbe d'évolution de votre erreur (_loss_) d'entraînement. Au long de votre entraînement, validez la performance de votre modèle sur l'ensemble de validation et affichez la _loss_ et l'_accuracy_ en fonction de l'époque.

Faites le suivi de l'_accuracy_ et identifiez la plus haute valeur obtenue au cours de l'entraînement. Faites une sauvegarde du modèle lorsque vous obtenez cette meilleure _accuracy_.

N.B.: Pour cette question, vous n'avez pas besoin de faire une validation à chaque époque. Vous pouvez choisir un intervalle plus grand (e.g. toutes les 2 époques) pour diminuer le temps de calcul.

## Apprentissage par transfert

**Q1**. Étant données les différences marquées entre les caractéristiques des images d'ImageNet (des chats, des chiens, des autobus, des spatules, etc.) et les images OCT de la rétine, pensez-vous qu'utiliser un modèle pré-entraîné sur ImageNet va aider vos performances de classification? Expliquez votre raisonnement.

**Q2**. Affinez pendant au moins 10 époques un modèle pré-entraîné. Créez le en utilisant l'argument `pretrained=True`. Durant l'entraînement du modèle, faites une sauvegarde du modèle lorsque vous obtenez la meilleure _accuracy_. Affichez les courbes d'évolution de votre _loss_ d'entraînement, et de votre _loss_ et _accuracy_ de validation.

**Q3.** Comparez les performances de votre modèle pré-entraîné avec celles de votre modèle non pré-entraîné. Qu'en concluez-vous?

## Évaluation finale

**Q1**. Évaluez votre meilleur modèle sur l'ensemble de test. Affichez la _loss_ et l'_accuracy_ finale.

**Q2**. Commentez les performances de votre modèle sur l'ensemble de test et comparez-les à celles obtenues sur l'ensemble de validation.


**Q3**. Pourquoi est-il important de maintenir un ensemble de test séparé de l'ensemble de validation? Que se passerait-il si on n'avait que deux ensembles (entraînement et validation/test) plutôt que trois (entraînement, validation, test)?

**Q4**. Au sein de l’ensemble de test, identifiez

<!-- | Une image CNV correctement identifiée comme CNV | Une image CNV incorrectement identifiée comme DME | 
 -->


| Une image CNV correctement identifiée comme CNV | Une image CNV incorrectement identifiée comme DME | Une image CNV incorrectement identifiée comme Drusen | Une image CNV incorrectement identifiée comme normale |
|---|---|---|---|
| Une image DME incorrectement identifiée comme CNV | Une image DME correctement identifiée comme DME | Une image DME incorrectement identifiée comme Drusen | Une image DME incorrectement identifiée comme normale |
| Une image Drusen incorrectement identifiée comme CNV | Une image Drusen incorrectement identifiée comme DME | Une image Drusen correctement identifiée comme Drusen | Une image Drusen incorrectement identifiée comme normale |
| Une image normale incorrectement identifiée comme CNV | Une image normale incorrectement identifiée comme DME | Une image normale incorrectement identifiée comme Drusen | Une image normale correctement identifiée comme normale |

**Q5**. Au vu des performances de votre meilleur modèle sur les données de test, seriez-vous à l'aise d'intégrer ce modèle dans des protocoles cliniques pour aider au diagnostic? Justifiez votre réponse.

## Visualisation de l'espace latent

**Q1**. Utilisez la fonction `get_embeddings` pour extraire les _vecteurs latents_ des images de l'ensemble de test issus
1. d'un modèle aléatoirement initialisé;
2. d'un modèle pré-entraîné sur ImageNet, mais non affiné;
3. de votre modèle entraîné (sans pré-entraînement);
4. de votre modèle affiné (avec pré-entraînement).

**Q2**. Pour chacun des trois ensembles de vecteurs latents obtenus à la question précédente, utilisez la méthode t-SNE pour réduire la dimensionnalité des vecteurs à 2D. Affichez les résultats dans un graphique de dispersion (_scatter plot_), où chaque point représente une image, et où la couleur du point indique la classe de l'image. Commentez vos observations.